Education Outcomes and Inequality: A Global Analysis of Completion Rates, Education Spending, and Gender Gaps


### 0. Imports and Setup


In [ ]:
#pip list --format=freeze > requirements.txt

In [1]:
import zipfile

with zipfile.ZipFile("./data/EdStats_CSV.zip", 'r') as zip:
    zip.extractall("./data")

In [1]:
import pandas as pd
import numpy as np
import sklearn
import seaborn as sns
import scipy
import plotly
import statsmodels
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
np.random.seed(SEED)

DATE_RANGE_INDEX = range(1999,2016)

In [2]:
df_data = pd.read_csv("./data/EdStatsData.csv")

df_country_series = pd.read_csv("./data/EdStatsCountry-Series.csv")
df_country = pd.read_csv("./data/EdStatsCountry.csv")

df_footnote= pd.read_csv("./data/EdStatsFootNote.csv")
df_series = pd.read_csv("./data/EdStatsSeries.csv")


In [3]:
# Only the columns from 1999 to 2015 seem well documented. 

df_data.drop("Unnamed: 69", axis=1, inplace=True)

cols_to_drop = [
    col for col in df_data.select_dtypes(exclude="O").columns
    if isinstance(col, str) and (int(col) < min(DATE_RANGE_INDEX) or int(col) > max(DATE_RANGE_INDEX))
        
]
df_data.drop(cols_to_drop, axis=1, inplace=True)

In [ ]:
# Intresting Indicator:
# SE.XPD.CUR.TOTL.ZS -> Current expenditure as % of total expenditure in public institutions (%)

# UIS.XSPENDP.0.FDPUB.FNCAP -> Capital expenditure as % of total expenditure in pre-primary public institutions (%)
# UIS.XSPENDP.1.FDPUB.FNCAP -> Capital expenditure as % of total expenditure in primary public institutions (%)

# UIS.XSPENDP.FDPUB.FNCAP 
# UIS.XSPENDP.23.FDPUB.FNCUR -> Current expenditure as % of total expenditure in secondary public institutions (%)
# UIS.XSPENDP.FDPUB.FNNONS -> Current expenditure other than staff compensation as % of total expenditure in public institutions (%)
# UIS.E.0.T -> Enrolment in early childhood education, both sexes (number)
# UIS.SAP.1.G1.M -> Population of the official entrance age to primary education, male (number)
# UIS.SAP.1.G1.F -> Population of the official entrance age to primary education, female (number)
# UIS.SLE.1.GPI -> School life expectancy, primary, gender parity index (GPI)
# UIS.NER.1.GPI -> Net enrolment rate, primary, gender parity index (GPI)
# UIS.NERA.1.GPI -> Adjusted net enrolment rate, primary, gender parity index (GPI)
# SP.PRM.GRAD.MA -> Population of the official age for the last grade of primary education, male (number)
# SP.PRM.GRAD.FE -> Population of the official age for the last grade of primary education, female (number)

In [ ]:
# Chosen Indicatos:
# SE.XPD.TOTL.GD.ZS -> Government expenditure on education as % of GDP (%)
# SE.XPD.TOTL.GB.ZS -> Expenditure on education as % of total government expenditure (%)

# NY.GDP.MKTP.PP.KD -> GDP, PPP (constant 2011 international $)
# NY.GDP.PCAP.PP.KD -> GDP per capita, PPP (constant 2011 international $)

# UIS.SAP.1.G1.M -> Population of the official entrance age to primary education, male (number)
# UIS.SAP.1.G1.F -> Population of the official entrance age to primary education, female (number)

# UIS.GER.123.F -> Gross enrolment ratio, primary and secondary, females (%)
# UIS.GER.123.M -> Gross enrolment ratio, primary and secondary, males (%)

# UIS.NER.1.GPI -> Net enrolment rate, primary, gender parity index (GPI)
# UIS.NER.23.GPI -> Net enrolment rate, secondary, gender parity index (GPI)

# SE.PRM.CMPT.FE.ZS -> Primary completion rate, female (%)
# SE.PRM.CMPT.MA.ZS -> Primary completion rate, male (%)

In [49]:
selected_codes = [
    "SE.XPD.TOTL.GD.ZS",
    "SE.XPD.TOTL.GB.ZS",
    "NY.GDP.MKTP.PP.KD",
    "NY.GDP.PCAP.PP.KD",
    "UIS.SAP.1.G1.M",
    "UIS.SAP.1.G1.F",
    "UIS.GER.123.F",
    "UIS.GER.123.M",
    "UIS.NER.1.GPI",
    "UIS.NER.23.GPI",
    "SE.PRM.CMPT.FE.ZS",
    "SE.PRM.CMPT.MA.ZS"]

In [ ]:
df_series[df_series["Series Code"].str.contains("UIS.GER.", case=False)]

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
3003,UIS.GER.0.GPI,Pre-Primary,"Gross enrolment ratio, pre-primary, gender par...",NaN,Ratio of female gross enrolment ratio for pre-...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3004,UIS.GER.12,Secondary,"Gross enrolment ratio, primary and lower secon...",NaN,Total enrollment in primary and lower secondar...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3005,UIS.GER.12.F,Secondary,"Gross enrolment ratio, primary and lower secon...",NaN,Total female enrollment in primary and lower s...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3006,UIS.GER.12.GPI,Secondary,"Gross enrolment ratio, primary and lower secon...",NaN,Ratio of female gross enrolment ratio for prim...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3007,UIS.GER.12.M,Secondary,"Gross enrolment ratio, primary and lower secon...",NaN,Total male enrollment in primary and lower sec...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3008,UIS.GER.123,Secondary,"Gross enrolment ratio, primary and secondary, ...",NaN,Total enrollment in primary and secondary educ...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3009,UIS.GER.123.F,Secondary,"Gross enrolment ratio, primary and secondary, ...",NaN,Total female enrollment in primary and seconda...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3010,UIS.GER.123.M,Secondary,"Gross enrolment ratio, primary and secondary, ...",NaN,Total male enrollment in primary and secondary...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3011,UIS.GER.1t6.F,Tertiary,"Gross enrolment ratio, primary to tertiary, fe...",NaN,"Total female enrollment in primary, secondary ...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3012,UIS.GER.1t6.GPI,Tertiary,"Gross enrolment ratio, primary to tertiary, ge...",NaN,Ratio of female gross enrolment ratio for prim...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#Ratio of female net enrolment rate for primary to the male net enrolment rate for primary. It is calculated by dividing the female value for the indicator by the male value for the indicator. A GPI equal to 1 indicates parity between females and males. In general, a value less than 1 indicates disparity in favor of males and a value greater than 1 indicates disparity in favor of females.

In [10]:
year_columns = [str(year) for year in DATE_RANGE_INDEX]
year_columns

['1999',
 '2000',
 '2001',
 '2002',
 '2003',
 '2004',
 '2005',
 '2006',
 '2007',
 '2008',
 '2009',
 '2010',
 '2011',
 '2012',
 '2013',
 '2014',
 '2015']

In [75]:
df_indicator_data = df_data.loc[(df_data["Indicator Code"].isin(selected_codes))]
df_indicator_data

,Country Name,Country Code,Indicator Name,Indicator Code,1999,2000,2001,2002,2003,2004,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
1224,Arab World,ARB,Expenditure on education as % of total governm...,SE.XPD.TOTL.GB.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1246,Arab World,ARB,"GDP per capita, PPP (constant 2011 internation...",NY.GDP.PCAP.PP.KD,1.116354e+04,1.145086e+04,1.149000e+04,1.142150e+04,1.159594e+04,1.240723e+04,...,1.343403e+04,1.386042e+04,1.437783e+04,1.435481e+04,1.475905e+04,1.482591e+04,1.534280e+04,1.545497e+04,NaN,NaN
1248,Arab World,ARB,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,3.035511e+12,3.178317e+12,3.257336e+12,3.308590e+12,3.433512e+12,3.757216e+12,...,4.260279e+12,4.500388e+12,4.779549e+12,4.882776e+12,5.132269e+12,5.265230e+12,5.561251e+12,5.714653e+12,NaN,NaN
1260,Arab World,ARB,Government expenditure on education as % of GD...,SE.XPD.TOTL.GD.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1324,Arab World,ARB,"Gross enrolment ratio, primary and secondary, ...",UIS.GER.123.F,6.828204e+01,6.957533e+01,7.106753e+01,7.260209e+01,7.407589e+01,7.540091e+01,...,7.694061e+01,7.665801e+01,7.747315e+01,7.841426e+01,7.945959e+01,8.052322e+01,8.216338e+01,8.091618e+01,81.176811,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
884921,Zimbabwe,ZWE,"Net enrolment rate, secondary, gender parity i...",UIS.NER.23.GPI,9.074100e-01,9.008000e-01,9.055300e-01,8.999500e-01,9.234700e-01,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.002280e+00,1.011080e+00,NaN,NaN
885601,Zimbabwe,ZWE,Population of the official entrance age to pri...,UIS.SAP.1.G1.F,1.782120e+05,1.775590e+05,1.775790e+05,1.772910e+05,1.769650e+05,1.769020e+05,...,1.784010e+05,1.803770e+05,1.832270e+05,1.868440e+05,1.910540e+05,1.957320e+05,2.015990e+05,2.082110e+05,215118.000000,221749.0
885602,Zimbabwe,ZWE,Population of the official entrance age to pri...,UIS.SAP.1.G1.M,1.795560e+05,1.787890e+05,1.789740e+05,1.786890e+05,1.782540e+05,1.780210e+05,...,1.793050e+05,1.813000e+05,1.841870e+05,1.878030e+05,1.919210e+05,1.965500e+05,2.024060e+05,2.090290e+05,215964.000000,222623.0
885801,Zimbabwe,ZWE,"Primary completion rate, female (%)",SE.PRM.CMPT.FE.ZS,8.778414e+01,NaN,8.699248e+01,8.177285e+01,8.050180e+01,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,9.035918e+01,9.093144e+01,NaN,NaN


In [67]:
df_indicator_data["Indicator Code"].value_counts()

Indicator Code
SE.XPD.TOTL.GB.ZS    242
NY.GDP.PCAP.PP.KD    242
NY.GDP.MKTP.PP.KD    242
SE.XPD.TOTL.GD.ZS    242
UIS.GER.123.F        242
UIS.GER.123.M        242
UIS.NER.1.GPI        242
UIS.NER.23.GPI       242
UIS.SAP.1.G1.F       242
UIS.SAP.1.G1.M       242
SE.PRM.CMPT.FE.ZS    242
SE.PRM.CMPT.MA.ZS    242
Name: count, dtype: int64

In [48]:
#df_data[df_data["Country Code"] == "USA"]
df_filled_data = df_data.loc[(df_data[year_columns].notna().all(axis=1)) & (df_data["Country Code"] == "USA") & (df_data["Indicator Name"].str.contains("gdp", case=False))]
df_filled_data

,Country Name,Country Code,Indicator Name,Indicator Code,1999,2000,2001,2002,2003,2004,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
847857,United States,USA,GDP at market prices (constant 2005 US$),NY.GDP.MKTP.KD,1.221327e+13,1.271306e+13,1.283714e+13,1.306642e+13,1.343317e+13,1.394171e+13,...,1.479230e+13,1.505540e+13,1.501149e+13,1.459484e+13,1.496437e+13,1.520402e+13,1.554216e+13,1.580286e+13,1.617746e+13,1.664039e+13
847858,United States,USA,GDP at market prices (current US$),NY.GDP.MKTP.CD,9.660624e+12,1.028478e+13,1.062182e+13,1.097751e+13,1.151067e+13,1.227493e+13,...,1.385589e+13,1.447764e+13,1.471858e+13,1.441874e+13,1.496437e+13,1.551793e+13,1.615526e+13,1.669152e+13,1.739310e+13,1.812071e+13
847859,United States,USA,GDP per capita (constant 2005 US$),NY.GDP.PCAP.KD,4.376888e+04,4.505582e+04,4.504749e+04,4.542865e+04,4.630404e+04,4.761428e+04,...,4.957540e+04,4.997953e+04,4.936464e+04,4.757561e+04,4.837388e+04,4.878347e+04,4.949759e+04,4.997663e+04,5.078252e+04,5.185591e+04
847860,United States,USA,GDP per capita (current US$),NY.GDP.PCAP.CD,3.462093e+04,3.644986e+04,3.727362e+04,3.816604e+04,3.967720e+04,4.192181e+04,...,4.643707e+04,4.806154e+04,4.840143e+04,4.700156e+04,4.837388e+04,4.979067e+04,5.145012e+04,5.278703e+04,5.459855e+04,5.646901e+04
847861,United States,USA,"GDP per capita, PPP (constant 2011 internation...",NY.GDP.PCAP.PP.KD,4.467255e+04,4.598605e+04,4.597755e+04,4.636658e+04,4.726004e+04,4.859734e+04,...,5.059895e+04,5.101143e+04,5.038384e+04,4.855787e+04,4.937262e+04,4.979067e+04,5.051953e+04,5.100846e+04,5.183099e+04,5.292654e+04
847862,United States,USA,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,3.462093e+04,3.644986e+04,3.727362e+04,3.816604e+04,3.967720e+04,4.192181e+04,...,4.643707e+04,4.806154e+04,4.840143e+04,4.700156e+04,4.837388e+04,4.979067e+04,5.145012e+04,5.278703e+04,5.459855e+04,5.646901e+04
847863,United States,USA,"GDP, PPP (constant 2011 international $)",NY.GDP.MKTP.PP.KD,1.246543e+13,1.297554e+13,1.310217e+13,1.333620e+13,1.371051e+13,1.422956e+13,...,1.509771e+13,1.536623e+13,1.532142e+13,1.489617e+13,1.527333e+13,1.551793e+13,1.586305e+13,1.612913e+13,1.651146e+13,1.698395e+13
847864,United States,USA,"GDP, PPP (current international $)",NY.GDP.MKTP.PP.CD,9.660624e+12,1.028478e+13,1.062182e+13,1.097751e+13,1.151067e+13,1.227493e+13,...,1.385589e+13,1.447764e+13,1.471858e+13,1.441874e+13,1.496437e+13,1.551793e+13,1.615526e+13,1.669152e+13,1.739310e+13,1.812071e+13


In [15]:
# SP.POP.1018.FE.UN = Population, ages 10-18, female
# SP.POP.1018.MA.UN = Population, ages 10-18, male
df_data[df_data["Country Code"] == "USA"]
df_filled_data = df_data.loc[(df_data["1999"] >= 1 ) & (df_data["Country Code"] == "USA") & (df_data["Indicator Name"].str.contains("male", case=False))]
df_filled_data

,Country Name,Country Code,Indicator Name,Indicator Code,1999,2000,2001,2002,2003,2004,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
846620,United States,USA,"Adjusted net enrolment rate, primary, female (%)",SE.PRM.TENR.FE,9.779105e+01,9.773678e+01,98.037827,9.679218e+01,96.170967,9.514385e+01,...,9.725755e+01,9.785545e+01,9.794343e+01,9.709736e+01,95.138184,9.467457e+01,9.410391e+01,9.381774e+01,9.443206e+01,94.765541
846622,United States,USA,"Adjusted net enrolment rate, primary, male (%)",SE.PRM.TENR.MA,9.725062e+01,9.750291e+01,96.620857,9.525357e+01,94.931084,9.376343e+01,...,9.529584e+01,9.652141e+01,9.704943e+01,9.501399e+01,94.468750,9.324649e+01,9.411461e+01,9.383361e+01,9.365279e+01,94.158813
846628,United States,USA,Adjusted net intake rate to Grade 1 of primary...,UIS.NIRA.1.F,8.803209e+01,8.423761e+01,85.486191,8.337826e+01,84.070671,7.939470e+01,...,8.290853e+01,8.766076e+01,9.020750e+01,8.369993e+01,82.832977,8.687901e+01,7.739850e+01,7.790285e+01,7.978172e+01,NaN
846630,United States,USA,Adjusted net intake rate to Grade 1 of primary...,UIS.NIRA.1.M,8.514682e+01,8.676923e+01,78.982811,7.753821e+01,81.926620,7.704565e+01,...,7.985333e+01,8.265864e+01,8.379000e+01,7.682121e+01,82.588860,8.253268e+01,7.760338e+01,7.564341e+01,7.733834e+01,NaN
847737,United States,USA,"Enrolment in early childhood education, female...",UIS.E.0.F,3.446666e+06,3.494559e+06,NaN,3.797312e+06,NaN,3.544902e+06,...,3.587208e+06,3.625338e+06,3.494297e+06,3.490333e+06,NaN,4.346524e+06,4.416691e+06,4.263450e+06,4.300430e+06,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
849704,United States,USA,"Total inbound internationally mobile students,...",UIS.MS.56.F,1.893600e+05,1.990960e+05,199096.000000,2.555930e+05,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.593570e+05,3.815390e+05,NaN
849710,United States,USA,"Total net enrolment rate, primary, female (%)",UIS.NERT.1.F,9.969056e+01,9.943364e+01,100.000000,9.821479e+01,97.547127,9.730418e+01,...,9.947161e+01,1.000000e+02,9.982290e+01,9.921085e+01,97.704102,9.707460e+01,9.693871e+01,9.596595e+01,9.802072e+01,NaN
849712,United States,USA,"Total net enrolment rate, primary, male (%)",UIS.NERT.1.M,9.952068e+01,9.946889e+01,98.953949,9.791414e+01,96.911438,9.643413e+01,...,9.800125e+01,9.932148e+01,9.970499e+01,9.774106e+01,97.430099,9.592931e+01,9.764005e+01,9.687037e+01,9.765747e+01,NaN
849961,United States,USA,"Unemployment, female (% of female labor force)",SL.UEM.TOTL.FE.ZS,4.300000e+00,4.100000e+00,4.600000,5.600000e+00,5.700000,5.400000e+00,...,4.600000e+00,4.500000e+00,5.400000e+00,8.100000e+00,8.600000,8.400000e+00,7.900000e+00,7.100000e+00,6.100000e+00,5.200000


In [28]:
df_data[df_data["Country Code"] == "USA"]
df_filled_data = df_data.loc[(df_data["2000"] >= 0) & (df_data["Country Code"] == "USA") & (df_data["Indicator Code"].str.contains("uis.ger", case=False))]
df_filled_data

,Country Name,Country Code,Indicator Name,Indicator Code,1999,2000,2001,2002,2003,2004,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
847924,United States,USA,"Gross enrolment ratio, lower secondary, gender...",UIS.GER.2.GPI,NaN,0.995930,0.996270,0.981290,1.001160,1.044020,...,0.993210,1.009030,0.998390,1.005550,1.014270,0.995430,1.009000,0.987660,1.004710,NaN
847926,United States,USA,"Gross enrolment ratio, post-secondary non-tert...",UIS.GER.4,42.444180,39.331379,41.450119,11.204090,10.163860,10.207100,...,10.132980,10.399710,9.808620,10.871180,12.344760,12.990500,12.587420,26.171120,25.937309,NaN
847927,United States,USA,"Gross enrolment ratio, post-secondary non-tert...",UIS.GER.4.F,50.338001,47.387451,49.007561,14.084080,13.874940,14.475730,...,14.358350,14.720400,13.723610,15.260180,16.808399,17.650419,17.545521,35.092339,34.553951,NaN
847928,United States,USA,"Gross enrolment ratio, post-secondary non-tert...",UIS.GER.4.GPI,1.437060,1.492870,1.427670,1.659790,2.082230,2.338870,...,2.320650,2.316070,2.237520,2.264210,2.064790,2.050570,2.213450,1.973230,1.938050,NaN
847929,United States,USA,"Gross enrolment ratio, post-secondary non-tert...",UIS.GER.4.M,35.028419,31.742590,34.326839,8.485440,6.663510,6.189190,...,6.187200,6.355780,6.133400,6.739730,8.140490,8.607570,7.926780,17.784220,17.829241,NaN
847932,United States,USA,"Gross enrolment ratio, pre-primary, gender par...",UIS.GER.0.GPI,0.966720,1.013320,NaN,1.028150,NaN,0.953320,...,0.997150,0.972700,0.984670,0.980560,1.032870,0.969440,0.968170,0.995640,1.004830,NaN
847934,United States,USA,"Gross enrolment ratio, primary and lower secon...",UIS.GER.12,102.970161,102.138702,102.623451,100.746529,101.377220,101.168053,...,101.581306,102.296333,102.507919,101.757637,100.181252,99.608780,99.707680,100.092850,100.337173,NaN
847935,United States,USA,"Gross enrolment ratio, primary and lower secon...",UIS.GER.12.F,NaN,101.599182,102.553932,100.809158,101.572617,101.876488,...,101.784843,102.438293,102.640259,102.082489,100.146378,99.703690,99.326881,99.536377,100.213760,NaN
847936,United States,USA,"Gross enrolment ratio, primary and lower secon...",UIS.GER.12.GPI,NaN,0.989750,0.998680,1.001210,1.003760,1.013730,...,1.003910,1.002710,1.002520,1.006250,0.999320,1.001860,0.992560,0.989180,0.997590,NaN
847937,United States,USA,"Gross enrolment ratio, primary and lower secon...",UIS.GER.12.M,NaN,102.650887,102.689468,100.687119,101.192001,100.496468,...,101.387970,102.161392,102.382057,101.448540,100.214447,99.518311,100.071404,100.625221,100.455368,NaN


In [16]:
range_ = np.arange(min(DATE_RANGE_INDEX), max(DATE_RANGE_INDEX)+1)
range_

array([1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009,
       2010, 2011, 2012, 2013, 2014, 2015])

In [ ]:
year_columns = [str(year) for year in range_]
year_columns


['1999',
 '2000',
 '2001',
 '2002',
 '2003',
 '2004',
 '2005',
 '2006',
 '2007',
 '2008',
 '2009',
 '2010',
 '2011',
 '2012',
 '2013',
 '2014',
 '2015']

In [6]:
year_columns = [str(year) for year in range_]

plot_data = (
    df_filled_data
    .melt(
        id_vars=["Indicator Code"],
        value_vars=year_columns,
        var_name="Year",
        value_name="Value"
    )
    .dropna(subset=["Value"])
)

plot_data["Year"] = plot_data["Year"].astype(int)

In [34]:
# UIS.GER.123 -> Gross enrolment ratio, primary and secondary, both sexes (%)
# UIS.GER.1t6.GPI -> Gross enrolment ratio, primary to tertiary, gender parity index (GPI)

df_series[df_series["Series Code"].str.contains("SE.XPD", case=False)]
#Total enrollment in primary and secondary education, regardless of age, expressed as a percentage of the total population of official primary and secondary education age. GER can exceed 100% due to the inclusion of over-aged and under-aged students because of early or late school entrance and grade repetition.


,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
2373,SE.XPD.CUR.TOTL.ZS,Expenditures,Current expenditure as % of total expenditure ...,Current expenditure expressed as a percentage ...,Current expenditure expressed as a percentage ...,NaN,NaN,NaN,All Levels,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2374,SE.XPD.PRIM.PC.ZS,Expenditures,Government expenditure per primary student as ...,"Average total (current, capital and transfers)...","Average total (current, capital and transfers)...",NaN,NaN,NaN,Primary,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2375,SE.XPD.PRIM.ZS,Expenditures,Expenditure on primary as % of government expe...,Expenditure on education by level of education...,Expenditure on education by level of education...,NaN,NaN,NaN,Primary,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2376,SE.XPD.SECO.PC.ZS,Expenditures,Government expenditure per secondary student a...,"Average total (current, capital and transfers)...","Average total (current, capital and transfers)...",NaN,NaN,NaN,Secondary,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2377,SE.XPD.SECO.ZS,Expenditures,Expenditure on secondary as % of government ex...,Expenditure on education by level of education...,Expenditure on education by level of education...,NaN,NaN,NaN,Secondary,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2378,SE.XPD.TERT.PC.ZS,Expenditures,Government expenditure per tertiary student as...,"Average total (current, capital and transfers)...","Average total (current, capital and transfers)...",NaN,NaN,NaN,Tertiary,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2379,SE.XPD.TERT.ZS,Expenditures,Expenditure on tertiary as % of government exp...,Expenditure on education by level of education...,Expenditure on education by level of education...,NaN,NaN,NaN,Tertiary,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2380,SE.XPD.TOTL.GB.ZS,Expenditures,Expenditure on education as % of total governm...,NaN,"Total general (local, regional and central) go...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2381,SE.XPD.TOTL.GD.ZS,Expenditures,Government expenditure on education as % of GD...,"Total general (local, regional and central) go...","Total general (local, regional and central) go...",NaN,NaN,NaN,All Levels,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_series[df_series["Series Code"].str.contains("PTRHC", case=False)]
#Average number of pupils per teacher at a given level of education, based on headcounts of both pupils and teachers. Divide the total number of pupils enrolled at the specified level of education by the number of teachers at the same level. In computing and interpreting this indicator, one should take into account the existence of part-time teaching, school-shifts, multi-grade classes and other practices that may affect the precision and meaningfulness of pupil-teacher ratios. When feasible, the number of part-time teachers is converted to ‘full-time equivalent’ teachers; a double-shift teacher is counted twice, etc. Teachers are defined as persons whose professional activity involves the transmitting of knowledge, attitudes and skills that are stipulated in a formal curriculum programme to students enrolled in a formal educational institution.

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
3273,UIS.PTRHC.02.QUALIFIED,Teachers,Pupil/qualified teacher ratio in pre-primary e...,NaN,Average number of pupils per qualified teacher...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3274,UIS.PTRHC.02.TRAINED,Teachers,Pupil/trained teacher ratio in pre-primary edu...,NaN,Average number of pupils per trained teacher a...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3275,UIS.PTRHC.1.QUALIFIED,Teachers,Pupil/qualified teacher ratio in primary educa...,NaN,Average number of pupils per qualified teacher...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3276,UIS.PTRHC.1.TRAINED,Teachers,Pupil/trained teacher ratio in primary educati...,NaN,Average number of pupils per trained teacher a...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3277,UIS.PTRHC.2,Teachers,Pupil-teacher ratio in lower secondary educati...,NaN,Average number of pupils per teacher at a give...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3278,UIS.PTRHC.2.QUALIFIED,Teachers,Pupil/qualified teacher ratio in lower seconda...,NaN,Average number of pupils per qualified teacher...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3279,UIS.PTRHC.2.TRAINED,Teachers,Pupil/trained teacher ratio in lower secondary...,NaN,Average number of pupils per trained teacher a...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3280,UIS.PTRHC.2T3.QUALIFIED,Teachers,Pupil/qualified teacher ratio in secondary (he...,NaN,Average number of pupils per qualified teacher...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3281,UIS.PTRHC.2T3.TRAINED,Teachers,Pupil/trained teacher ratio in secondary educa...,NaN,Average number of pupils per trained teacher a...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3282,UIS.PTRHC.3,Teachers,Pupil-teacher ratio in upper secondary educati...,NaN,Average number of pupils per teacher at a give...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 886930 entries, 0 to 886929
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Country Name    886930 non-null  str    
 1   Country Code    886930 non-null  str    
 2   Indicator Name  886930 non-null  str    
 3   Indicator Code  886930 non-null  str    
 4   1999            118839 non-null  float64
 5   2000            176676 non-null  float64
 6   2001            123509 non-null  float64
 7   2002            124205 non-null  float64
 8   2003            130363 non-null  float64
 9   2004            128814 non-null  float64
 10  2005            184108 non-null  float64
 11  2006            140312 non-null  float64
 12  2007            137272 non-null  float64
 13  2008            134387 non-null  float64
 14  2009            142108 non-null  float64
 15  2010            242442 non-null  float64
 16  2011            146012 non-null  float64
 17  2012            14726

In [10]:
display(df_data["Indicator Code"].value_counts()) # 10*242*366+5*242 = 886930
display(df_data["Country Code"].value_counts())
display(df_data["Country Name"].value_counts())

Indicator Code
UIS.NERA.2              242
UIS.NERA.2.F            242
UIS.NERA.2.GPI          242
UIS.NERA.2.M            242
SE.PRM.TENR             242
                       ... 
UIS.LP.AG15T24.M        242
SE.ADT.1524.LT.ZS       242
SE.ADT.1524.LT.FE.ZS    242
SE.ADT.1524.LT.FM.ZS    242
SE.ADT.1524.LT.MA.ZS    242
Name: count, Length: 3665, dtype: int64

Country Code
ARB    3665
EAS    3665
EAP    3665
EMU    3665
ECS    3665
       ... 
VIR    3665
PSE    3665
YEM    3665
ZMB    3665
ZWE    3665
Name: count, Length: 242, dtype: int64

Country Name
Arab World                                     3665
East Asia & Pacific                            3665
East Asia & Pacific (excluding high income)    3665
Euro area                                      3665
Europe & Central Asia                          3665
                                               ... 
Virgin Islands (U.S.)                          3665
West Bank and Gaza                             3665
Yemen, Rep.                                    3665
Zambia                                         3665
Zimbabwe                                       3665
Name: count, Length: 242, dtype: int64

In [ ]:
df_data_eu = df_data[df_data["CountryCode"] == "CHI"]
display(country_chi)

In [ ]:
df_footnote["SeriesCode"].value_counts()

SeriesCode
SH.DYN.MORT             9226
SE.PRM.AGES             8771
SE.PRM.DURS             8771
SE.SEC.DURS             8619
SE.SEC.AGES             8581
                        ... 
LO.TIMSS.MAT4.HI.MA        1
LO.TIMSS.MAT4.INT.FE       1
LO.TIMSS.MAT4.INT.MA       1
LO.TIMSS.MAT4.LOW.FE       1
LO.TIMSS.MAT4.BL.FE        1
Name: count, Length: 1558, dtype: int64

In [11]:
df_series.sample(10)

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
1960,PRJ.POP.2529.1.MF,Attainment,Projection: Population age 25-29 in thousands ...,NaN,Total population in thousands in the specified...,NaN,NaN,Projections (2010 to 2100),NaN,NaN,...,NaN,NaN,Wittgenstein Centre for Demography and Global ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3186,UIS.NART.2.Q2.GPI,Education Equality,"UIS: Total net attendance rate, lower secondar...",Ratio of female total net attendance rate to m...,Ratio of female total net attendance rate to m...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2813,UIS.E.2.GPV.GUK.F,Secondary,Enrolment in lower secondary general education...,NaN,Total number of female students enrolled in lo...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1104,LO.LLECE.REA6.2.FE,Learning Outcomes,LLECE: Female 6th grade students by reading pr...,Percentage of 6th grade female students scorin...,Percentage of 6th grade female students scorin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Latin American Laboratory for Assessment of th...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2942,UIS.FEP.56.FUK,Tertiary,Percentage of students enrolled in programmes ...,NaN,Total number of female students in unspecified...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
321,BAR.TER.ICMP.5559.ZS,Attainment,Barro-Lee: Percentage of population age 55-59 ...,Percentage of population age 55-59 with tertia...,Percentage of population age 55-59 with tertia...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1754,PRJ.ATT.2039.NED.MF,Attainment,Projection: Percentage of the population age 2...,NaN,Share of the population of the stated age grou...,NaN,NaN,Projections (2010 to 2100),NaN,NaN,...,NaN,NaN,Wittgenstein Centre for Demography and Global ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1462,LO.PISA.REA.MA,Learning Outcomes,PISA: Mean performance on the reading scale. Male,Average score of 15-year-old male students on ...,Average score of 15-year-old male students on ...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,OECD Programme for International Student Asses...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2470,SP.POP.1117.TO.UN,Population,"Population, ages 11-17, total","Population, ages 11-17, total is the total pop...","Population, ages 11-17, total is the total pop...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics (Derived),NaN,NaN,NaN,NaN,NaN,NaN,NaN
3332,UIS.REPP.1.G4.M,Primary,Percentage of repeaters in Grade 4 of primary ...,NaN,Total number of male pupils in the specified g...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Kurze Regelungen:
# - Nur Nominal und Ordinale Skalen werden beibehalten. -> Beschreibungen, Redundante Namen, Lizensen werden verworfen.
# - Nur Zeitachse von 1999 bis 2026 wird behalten
# - 
display("df_data:", df_data.columns) # Zeitliche Angaben von 1970 bis 2100 + Spalte 69. -> Als Relevant definier ich 2000 bis 2026, Rest wird verworfen.

display("df_country_series:", df_country_series.columns)
display("df_country:",df_country.columns)

display("df_footnote",df_footnote.columns)
display("df_series:",df_series.columns)

'df_data:'

Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007',
       '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015'],
      dtype='str')

'df_country_series:'

Index(['CountryCode', 'SeriesCode', 'DESCRIPTION', 'Unnamed: 3'], dtype='str')

'df_country:'

Index(['Country Code', 'Short Name', 'Table Name', 'Long Name', '2-alpha code',
       'Currency Unit', 'Special Notes', 'Region', 'Income Group', 'WB-2 code',
       'National accounts base year', 'National accounts reference year',
       'SNA price valuation', 'Lending category', 'Other groups',
       'System of National Accounts', 'Alternative conversion factor',
       'PPP survey year', 'Balance of Payments Manual in use',
       'External debt Reporting status', 'System of trade',
       'Government Accounting concept', 'IMF data dissemination standard',
       'Latest population census', 'Latest household survey',
       'Source of most recent Income and expenditure data',
       'Vital registration complete', 'Latest agricultural census',
       'Latest industrial data', 'Latest trade data',
       'Latest water withdrawal data', 'Unnamed: 31'],
      dtype='str')

'df_footnote'

Index(['CountryCode', 'SeriesCode', 'Year', 'DESCRIPTION', 'Unnamed: 4'], dtype='str')

'df_series:'

Index(['Series Code', 'Topic', 'Indicator Name', 'Short definition',
       'Long definition', 'Unit of measure', 'Periodicity', 'Base Period',
       'Other notes', 'Aggregation method', 'Limitations and exceptions',
       'Notes from original source', 'General comments', 'Source',
       'Statistical concept and methodology', 'Development relevance',
       'Related source links', 'Other web links', 'Related indicators',
       'License Type', 'Unnamed: 20'],
      dtype='str')

In [3]:
display(df_data.columns)
display(df_data.sample(10))
display(df_data.info())

Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978',
       '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987',
       '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996',
       '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005',
       '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014',
       '2015', '2016', '2017', '2020', '2025', '2030', '2035', '2040', '2045',
       '2050', '2055', '2060', '2065', '2070', '2075', '2080', '2085', '2090',
       '2095', '2100', 'Unnamed: 69'],
      dtype='str')

,Country Name,Country Code,Indicator Name,Indicator Code,1970,1971,1972,1973,1974,1975,...,2060,2065,2070,2075,2080,2085,2090,2095,2100,Unnamed: 69
471498,"Korea, Rep.",KOR,"Population, age 2, male",SP.POP.AG02.MA.UN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
589164,Myanmar,MMR,SABER: (School Finance) Policy Goal 4 Lever 2:...,SABER.SCH.FNNC.GOAL4.LVL2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25448,European Union,EUU,Wittgenstein Projection: Percentage of the pop...,PRJ.ATT.2039.3.MF,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
667853,Puerto Rico,PRI,EGRA: Correct Non-Words Read Per Minute (Mean)...,LO.EGRA.NCWPM.ICI.2GRD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53207,Lower middle income,LMC,Percentage of graduates from Agriculture progr...,SE.TER.GRAD.FE.AG.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
356247,Germany,DEU,EGRA: Correct Letter Names Read Per Minute (Me...,LO.EGRA.CLPM.AMH.2GRD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
567157,Moldova,MDA,SABER: (School Autonomy Accountability) Policy...,SABER.SCH.ATNM.GOAL4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
321958,Estonia,EST,"UIS: Adjusted net attendance rate, primary, fo...",UIS.NARA.1.Q4.GPI,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
755878,St. Lucia,LCA,EGRA: Listening Comprehension - Share of stude...,LO.EGRA.LSTN.0.ICI.2GRD,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
803262,Togo,TGO,DHS: Secondary completion rate. Rural,HH.DHS.SCR.R,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 886930 entries, 0 to 886929
Data columns (total 70 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Country Name    886930 non-null  str    
 1   Country Code    886930 non-null  str    
 2   Indicator Name  886930 non-null  str    
 3   Indicator Code  886930 non-null  str    
 4   1970            72288 non-null   float64
 5   1971            35537 non-null   float64
 6   1972            35619 non-null   float64
 7   1973            35545 non-null   float64
 8   1974            35730 non-null   float64
 9   1975            87306 non-null   float64
 10  1976            37483 non-null   float64
 11  1977            37574 non-null   float64
 12  1978            37576 non-null   float64
 13  1979            36809 non-null   float64
 14  1980            89122 non-null   float64
 15  1981            38777 non-null   float64
 16  1982            37511 non-null   float64
 17  1983            38460

None

In [18]:
display(df_footnote.columns)
display(df_footnote.sample(10))
display(df_footnote.info())

Index(['CountryCode', 'SeriesCode', 'Year', 'DESCRIPTION', 'Unnamed: 4'], dtype='str')

,CountryCode,SeriesCode,Year,DESCRIPTION,Unnamed: 4
101137,CMR,SE.PRM.TCHR.FE.ZS,YR2009,Estimated,NaN
627145,VUT,UIS.DR.2.GPV.G1.F,YR2003,UNESCO Institute for Statistics (UIS) estimate,NaN
599250,TZA,SE.SEC.ENRL,YR1986,Country Data,NaN
292808,KGZ,SE.PRM.CMPL.ZS,YR2011,Estimated,NaN
641981,ZMB,SE.TER.ENRR.FE,YR1982,Country Data,NaN
619565,VCT,UIS.T.23.V.F,YR2004,UNESCO Institute for Statistics (UIS) estimate,NaN
595173,TUV,SP.PRE.TOTL.FE.IN,YR2008,National Estimation,NaN
633839,WLD,UIS.SLE.56.GPI,YR2014,UNESCO Institute for Statistics (UIS) estimate,NaN
231615,HIC,SE.PRM.TCHR,YR1983,UIS Estimation,NaN
437411,MYS,SE.SCH.LIFE,YR1992,Country Data,NaN


<class 'pandas.DataFrame'>
RangeIndex: 643638 entries, 0 to 643637
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   CountryCode  643638 non-null  str    
 1   SeriesCode   643638 non-null  str    
 2   Year         643638 non-null  str    
 3   DESCRIPTION  643638 non-null  str    
 4   Unnamed: 4   0 non-null       float64
dtypes: float64(1), str(4)
memory usage: 24.6 MB


None

In [14]:
display(df_country.columns)
display(df_country.sample(10))
display(df_country.info())
display(df_country.describe())

Index(['Country Code', 'Short Name', 'Table Name', 'Long Name', '2-alpha code',
       'Currency Unit', 'Special Notes', 'Region', 'Income Group', 'WB-2 code',
       'National accounts base year', 'National accounts reference year',
       'SNA price valuation', 'Lending category', 'Other groups',
       'System of National Accounts', 'Alternative conversion factor',
       'PPP survey year', 'Balance of Payments Manual in use',
       'External debt Reporting status', 'System of trade',
       'Government Accounting concept', 'IMF data dissemination standard',
       'Latest population census', 'Latest household survey',
       'Source of most recent Income and expenditure data',
       'Vital registration complete', 'Latest agricultural census',
       'Latest industrial data', 'Latest trade data',
       'Latest water withdrawal data', 'Unnamed: 31'],
      dtype='str')

,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data,Unnamed: 31
67,ETH,Ethiopia,Ethiopia,Federal Democratic Republic of Ethiopia,ET,Ethiopian birr,Fiscal year end: July 7; reporting period for ...,Sub-Saharan Africa,Low income,ET,...,General Data Dissemination System (GDDS),2007,"Demographic and Health Survey (DHS), 2011","Expenditure survey/budget survey (ES/BS), 2010/11",NaN,NaN,2009.0,2012.0,2002,NaN
34,CHE,Switzerland,Switzerland,Switzerland,CH,Swiss franc,NaN,Europe & Central Asia,High income: OECD,CH,...,Special Data Dissemination Standard (SDDS),2010,NaN,"Expenditure survey/budget survey (ES/BS), 2000",Yes,2008,2007.0,2012.0,2000,NaN
172,PER,Peru,Peru,Republic of Peru,PE,Peruvian new sol,NaN,Latin America & Caribbean,Upper middle income,PE,...,Special Data Dissemination Standard (SDDS),2007,Continuous Demographic and Health Survey (DHS)...,"Integrated household survey (IHS), 2012",NaN,2012,2010.0,2012.0,2000,NaN
45,CRI,Costa Rica,Costa Rica,Republic of Costa Rica,CR,Costa Rican colon,NaN,Latin America & Caribbean,Upper middle income,CR,...,Special Data Dissemination Standard (SDDS),2011,"Multiple Indicator Cluster Survey (MICS), 2011","Integrated household survey (IHS), 2012",Yes,2014,2010.0,2012.0,1997,NaN
51,DEU,Germany,Germany,Federal Republic of Germany,DE,Euro,A simple multiplier is used to convert the nat...,Europe & Central Asia,High income: OECD,DE,...,Special Data Dissemination Standard (SDDS),2011,NaN,"Integrated household survey (IHS), 2000",Yes,2010,2009.0,2012.0,2007,NaN
83,GRC,Greece,Greece,Hellenic Republic,GR,Euro,A simple multiplier is used to convert the nat...,Europe & Central Asia,High income: OECD,GR,...,Special Data Dissemination Standard (SDDS),2011,NaN,"Integrated household survey (IHS), 2000",Yes,2009,2007.0,2012.0,2007,NaN
157,NAC,North America,North America,North America,XU,NaN,North America regional aggregate. There are no...,NaN,NaN,XU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
214,TJK,Tajikistan,Tajikistan,Republic of Tajikistan,TJ,Tajik somoni,NaN,Europe & Central Asia,Low income,TJ,...,General Data Dissemination System (GDDS),2010,"Demographic and Health Survey (DHS), 2012",Living Standards Measurement Study Survey (LSM...,NaN,2013,NaN,2000.0,2006,NaN
50,CZE,Czech Republic,Czech Republic,Czech Republic,CZ,Czech koruna,NaN,Europe & Central Asia,High income: OECD,CZ,...,Special Data Dissemination Standard (SDDS),2011,"World Health Survey (WHS), 2003","Income survey (IS), 1996",Yes,2010,2007.0,2012.0,2007,NaN
212,TGO,Togo,Togo,Republic of Togo,TG,CFA franc,"April 2013 database update: Based on IMF data,...",Sub-Saharan Africa,Low income,TG,...,General Data Dissemination System (GDDS),2010,"Demographic and Health Survey (DHS), 2013",Core Welfare Indicator Questionnaire Survey (C...,NaN,2011/2012,NaN,2012.0,2002,NaN


<class 'pandas.DataFrame'>
RangeIndex: 241 entries, 0 to 240
Data columns (total 32 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   Country Code                                       241 non-null    str    
 1   Short Name                                         241 non-null    str    
 2   Table Name                                         241 non-null    str    
 3   Long Name                                          241 non-null    str    
 4   2-alpha code                                       238 non-null    str    
 5   Currency Unit                                      215 non-null    str    
 6   Special Notes                                      145 non-null    str    
 7   Region                                             214 non-null    str    
 8   Income Group                                       214 non-null    str    
 9   WB-2 code            

None

,National accounts reference year,Latest industrial data,Latest trade data,Unnamed: 31
count,32.00000,107.000000,185.000000,0.0
mean,2001.53125,2008.102804,2010.994595,NaN
std,5.24856,2.616834,2.569675,NaN
min,1987.00000,2000.000000,1995.000000,NaN
25%,1996.75000,2007.500000,2011.000000,NaN
50%,2002.00000,2009.000000,2012.000000,NaN
75%,2005.00000,2010.000000,2012.000000,NaN
max,2012.00000,2010.000000,2012.000000,NaN


In [20]:
display(df_series.columns)
display(df_series.sample(10))
display(df_series.info())

attainment = df_series[df_series["Topic"] == "Attainment"]
attainment

# Fazit bisher: Alles unnötig bis Series Code und eventuell Topic

Index(['Series Code', 'Topic', 'Indicator Name', 'Short definition',
       'Long definition', 'Unit of measure', 'Periodicity', 'Base Period',
       'Other notes', 'Aggregation method', 'Limitations and exceptions',
       'Notes from original source', 'General comments', 'Source',
       'Statistical concept and methodology', 'Development relevance',
       'Related source links', 'Other web links', 'Related indicators',
       'License Type', 'Unnamed: 20'],
      dtype='str')

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
701,LO.EGRA.CWPM.ZERO.ARB.3GRD,Learning Outcomes,EGRA: Oral Reading Fluency - Share of students...,Percentage of students who were unable to read...,Percentage of students who were unable to read...,NaN,NaN,NaN,EGRA,NaN,...,NaN,NaN,Early Grade Reading Assessment (EGRA): https:/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3591,UIS.XGOVEXP.IMF.0,Expenditures,Expenditure on pre-primary as % of total gover...,NaN,"Total general (local, regional and central) go...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3514,UIS.SR.1.G4.M,Primary,"Survival rate to Grade 4 of primary education,...",NaN,Percentage of a cohort of male students enroll...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3241,UIS.NIR.1.AgM1.M,Primary,Net intake rate to Grade 1 of primary educatio...,NaN,Number of new male entrants in the first grade...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1020,LO.LLECE.MAT3.1.FE,Learning Outcomes,LLECE: Female 3rd grade students by mathematic...,Percentage of 3rd grade female students scorin...,Percentage of 3rd grade female students scorin...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Latin American Laboratory for Assessment of th...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2132,SABER.SCH.FNNC.GOAL5.LVL2,School Finance (SABER),SABER: (School Finance) Policy Goal 5 Lever 2:...,Data Interpretation: 1=Latent; 2=Emerging; 3=E...,Data Interpretation: 1=Latent; 2=Emerging; 3=E...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Systems Approach for Better Education Results ...,NaN,NaN,http://saber.worldbank.org/index.cfm,NaN,NaN,NaN,NaN
1974,PRJ.POP.2529.S1.MA,Attainment,Projection: Population age 25-29 in thousands ...,NaN,Total population in thousands in the specified...,NaN,NaN,Projections (2010 to 2100),NaN,NaN,...,NaN,NaN,Wittgenstein Centre for Demography and Global ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
598,HH.MICS.YRS.1519.R,Education Equality,MICS: Average years of schooling by age group....,Average years of schooling by age group. Age 1...,Average years of schooling by age group. Age 1...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,World Bank staff calculations based on Multipl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2607,SP.SEC.TOTL.FE.IN,Population,Population of the official age for secondary e...,NaN,Female population of the age-group theoretical...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
475,HH.DHS.YRS.1519.Q3,Education Equality,DHS: Average years of schooling by age group. ...,Average years of schooling by age group. Age 1...,Average years of schooling by age group. Age 1...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Demographic and Health Surveys (DHS),NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 3665 entries, 0 to 3664
Data columns (total 21 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Series Code                          3665 non-null   str    
 1   Topic                                3665 non-null   str    
 2   Indicator Name                       3665 non-null   str    
 3   Short definition                     2156 non-null   str    
 4   Long definition                      3665 non-null   str    
 5   Unit of measure                      0 non-null      float64
 6   Periodicity                          99 non-null     str    
 7   Base Period                          314 non-null    str    
 8   Other notes                          552 non-null    str    
 9   Aggregation method                   47 non-null     str    
 10  Limitations and exceptions           14 non-null     str    
 11  Notes from original source           0 no

None

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
0,BAR.NOED.1519.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 15-19 with...,Percentage of female population age 15-19 with...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BAR.NOED.1519.ZS,Attainment,Barro-Lee: Percentage of population age 15-19 ...,Percentage of population age 15-19 with no edu...,Percentage of population age 15-19 with no edu...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BAR.NOED.15UP.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 15+ with n...,Percentage of female population age 15+ with n...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BAR.NOED.15UP.ZS,Attainment,Barro-Lee: Percentage of population age 15+ wi...,Percentage of population age 15+ with no educa...,Percentage of population age 15+ with no educa...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BAR.NOED.2024.FE.ZS,Attainment,Barro-Lee: Percentage of female population age...,Percentage of female population age 20-24 with...,Percentage of female population age 20-24 with...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2909,UIS.EA.S1.AG25T99.F,Attainment,UIS: Percentage of population age 25+ with som...,The percentage of female population (age 25 an...,The percentage of female population (age 25 an...,NaN,NaN,NaN,Single Level Attainment/ Not Cumulative,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2910,UIS.EA.S1.AG25T99.M,Attainment,UIS: Percentage of population age 25+ with som...,The percentage of male population (age 25 and ...,The percentage of male population (age 25 and ...,NaN,NaN,NaN,Single Level Attainment/ Not Cumulative,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2911,UIS.EA.UK.AG25T99,Attainment,UIS: Percentage of population age 25+ with unk...,The percentage of the population (age 25 and o...,The percentage of the population (age 25 and o...,NaN,NaN,NaN,Single Level Attainment/ Not Cumulative,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2912,UIS.EA.UK.AG25T99.F,Attainment,UIS: Percentage of population age 25+ with unk...,The percentage of the female population (age 2...,The percentage of the female population (age 2...,NaN,NaN,NaN,Single Level Attainment/ Not Cumulative,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
display(df_country_series.columns)
display(df_country_series.sample(10))
display(df_country_series.info())

# Fazit bisher:

Index(['CountryCode', 'SeriesCode', 'DESCRIPTION', 'Unnamed: 3'], dtype='str')

,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
25,ARG,NY.GDP.MKTP.PP.CD,Estimates are based on regression.,NaN
129,CYP,NY.GDP.PCAP.KD,Data are for areas under the effective control...,NaN
458,SDN,SP.POP.GROW,Data sources : United Nations World Population...,NaN
231,HND,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
481,SOM,NY.GDP.MKTP.PP.KD,Estimates are based on regression.,NaN
496,SRB,SP.POP.TOTL.FE.ZS,Excluding Kosovo.,NaN
199,GHA,SP.POP.TOTL,Data sources : United Nations World Population...,NaN
354,MKD,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
100,CMR,SP.POP.TOTL,Data sources : United Nations World Population...,NaN
542,TLS,NY.GDP.MKTP.PP.KD,Estimates are based on regression.,NaN


<class 'pandas.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CountryCode  613 non-null    str    
 1   SeriesCode   613 non-null    str    
 2   DESCRIPTION  613 non-null    str    
 3   Unnamed: 3   0 non-null      float64
dtypes: float64(1), str(3)
memory usage: 19.3 KB


None

In [3]:
df_footnote.info()

<class 'pandas.DataFrame'>
RangeIndex: 643638 entries, 0 to 643637
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   CountryCode  643638 non-null  str    
 1   SeriesCode   643638 non-null  str    
 2   Year         643638 non-null  str    
 3   DESCRIPTION  643638 non-null  str    
 4   Unnamed: 4   0 non-null       float64
dtypes: float64(1), str(4)
memory usage: 24.6 MB


In [ ]:
df_series_code = df_footnote[df_footnote["SeriesCode"].isin(["SP.POP.GROW" , "NOED"])] #attainment
df_series_code

,CountryCode,SeriesCode,Year,DESCRIPTION,Unnamed: 4


In [9]:
country = df_country_series[df_country_series["CountryCode"] == "DEU"]
country

,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
132,DEU,SP.POP.GROW,Data sources : Eurostat,NaN
133,DEU,SP.POP.TOTL,Data sources : Eurostat,NaN


In [ ]:
df_series_code = df_series[df_series["Series Code"].isin(["SP.POP.GROW" , "SP.POP.TOTL"])] 
df_series_code

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
2589,SP.POP.GROW,Health: Population: Dynamics,Population growth (annual %),Annual population growth rate. Population is b...,Annual population growth rate for year t is th...,NaN,Annual,NaN,NaN,Weighted average,...,NaN,NaN,Derived from total population. Population sour...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2590,SP.POP.TOTL,Health: Population: Structure,"Population, total",NaN,Total population is based on the de facto defi...,NaN,Annual,NaN,NaN,Sum,...,NaN,Relevance to gender indicator: disaggregating ...,(1) United Nations Population Division. World ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
display(df_country_series["CountryCode"].value_counts(), sort=True)
# CHI und CHN könnte gleich sein
display(df_country_series["SeriesCode"].value_counts(), sort=True) # 


display(df_series["Series Code"].value_counts(), sort=True)


CountryCode
GEO    18
MDA    18
CYP    12
MAR    12
MUS    12
       ..
WSM     2
YEM     2
ZAF     2
ZMB     2
ZWE     2
Name: count, Length: 211, dtype: int64

SeriesCode
SP.POP.TOTL          211
SP.POP.GROW          211
NY.GDP.PCAP.PP.CD     19
NY.GDP.PCAP.PP.KD     19
NY.GNP.PCAP.PP.CD     19
NY.GDP.MKTP.PP.KD     14
NY.GNP.MKTP.PP.CD     14
NY.GDP.MKTP.PP.CD     14
SP.POP.1564.TO.ZS     13
SP.POP.TOTL.MA.ZS     13
SP.POP.TOTL.FE.ZS     13
SP.POP.0014.TO.ZS     13
NY.GNP.PCAP.CD         6
NY.GDP.PCAP.CD         5
NY.GDP.PCAP.KD         5
SP.POP.1564.MA.IN      4
SP.POP.0014.TO         4
SP.POP.1564.TO         4
SP.POP.1564.FE.IN      4
SP.POP.0014.MA.IN      4
SP.POP.0014.FE.IN      4
Name: count, dtype: int64

Series Code
BAR.NOED.1519.FE.ZS            1
BAR.NOED.1519.ZS               1
BAR.NOED.15UP.FE.ZS            1
BAR.NOED.15UP.ZS               1
BAR.NOED.2024.FE.ZS            1
                              ..
UIS.XUNIT.USCONST.3.FSGOV      1
UIS.XUNIT.USCONST.4.FSGOV      1
UIS.XUNIT.USCONST.56.FSGOV     1
XGDP.23.FSGOV.FDINSTADM.FFD    1
XGDP.56.FSGOV.FDINSTADM.FFD    1
Name: count, Length: 3665, dtype: int64

In [12]:
# CHI wird fuer Chile stehen und CHN fuer China. CN exestiert nicht.

country_chi = df_country_series[df_country_series["CountryCode"] == "CHI"]
display(country_chi)

country_chn = df_country_series[df_country_series["CountryCode"] == "CHL"]
display(country_chn)

,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
84,CHI,SP.POP.TOTL,Data sources : United Nations World Population...,NaN
85,CHI,SP.POP.0014.TO.ZS,"Refers to Guernsey, and Jersey.",NaN
86,CHI,SP.POP.1564.TO.ZS,"Refers to Guernsey, and Jersey.",NaN
87,CHI,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
88,CHI,SP.POP.TOTL.FE.ZS,"Refers to Guernsey, and Jersey.",NaN
89,CHI,SP.POP.TOTL.MA.ZS,"Refers to Guernsey, and Jersey.",NaN


,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
90,CHL,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
91,CHL,SP.POP.TOTL,Data sources : United Nations World Population...,NaN


In [13]:
df.isnull().sum()

NameError: name 'df' is not defined

In [ ]:
df.tail(20)

,Country Name,Country Code,Indicator Name,Indicator Code,1970,1971,1972,1973,1974,1975,...,2060,2065,2070,2075,2080,2085,2090,2095,2100,Unnamed: 69
886910,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.NED.FE,NaN,NaN,NaN,NaN,NaN,NaN,...,4.13,2.58,1.71,1.19,0.86,0.66,0.56,0.51,0.49,NaN
886911,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.NED.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,1.56,1.10,0.81,0.64,0.53,0.47,0.45,0.43,0.43,NaN
886912,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.NED.MF,NaN,NaN,NaN,NaN,NaN,NaN,...,5.69,3.68,2.53,1.82,1.39,1.14,1.01,0.94,0.91,NaN
886913,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.4.FE,NaN,NaN,NaN,NaN,NaN,NaN,...,576.34,636.47,699.49,764.63,832.26,903.02,978.19,1057.39,1139.21,NaN
886914,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.4.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,590.14,629.83,674.80,724.51,779.53,840.53,908.72,983.44,1063.26,NaN
886915,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.4.MF,NaN,NaN,NaN,NaN,NaN,NaN,...,1166.47,1266.30,1374.29,1489.14,1611.78,1743.55,1886.91,2040.83,2202.47,NaN
886916,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.1.FE,NaN,NaN,NaN,NaN,NaN,NaN,...,282.09,241.06,201.98,167.68,137.86,113.78,93.18,77.30,65.11,NaN
886917,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.1.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,212.14,179.40,150.11,126.43,106.82,91.70,78.21,67.56,59.06,NaN
886918,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.1.MF,NaN,NaN,NaN,NaN,NaN,NaN,...,494.23,420.47,352.09,294.10,244.68,205.47,171.39,144.86,124.17,NaN
886919,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.3.FE,NaN,NaN,NaN,NaN,NaN,NaN,...,3469.66,3493.32,3491.72,3468.64,3429.54,3378.61,3317.67,3248.91,3175.64,NaN


### 1. EDA


### 2. Feature Engineering


### 3. Modeling and Training


### 4. Evaluation
